<a href="https://colab.research.google.com/github/shreyanshsamaje7-prog/BIS_LAB/blob/main/Parallel_Cellular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np

# 3 x 3 agricultural grid
np.random.seed(10)

soil = np.random.uniform(20, 35, (3, 3))
target = np.full((3, 3), 40.0)

# Irrigation parameters
iterations = 30
eta = 0.6
neighbor_weight = 0.3
max_irrigation = 15
water_budget = 70
ET = 2  # Daily evapotranspiration

# Find neighboring cells (Von Neumann)
def get_neighbors(grid, i, j):
    values = []

    for di, dj in [(-1, 0), (1, 0),
                   (0, -1), (0, 1)]:

        ni, nj = i + di, j + dj

        if 0 <= ni < 3 and 0 <= nj < 3:
            values.append(grid[ni, nj])

    return np.mean(values) if values else grid[i, j]


# Parallel Cellular Algorithm
for t in range(iterations):

    old_soil = soil.copy()
    irrigation = np.zeros((3, 3))

    # Calculate irrigation for every cell
    for i in range(3):
        for j in range(3):

            neighbor = get_neighbors(old_soil, i, j)

            combined = (
                (1 - neighbor_weight) * old_soil[i, j]
                + neighbor_weight * neighbor
            )

            required = eta * (target[i, j] - combined) + ET

            irrigation[i, j] = np.clip(
                required, 0, max_irrigation
            )

    # Apply total water budget
    total_water = irrigation.sum()

    if total_water > water_budget:
        irrigation *= water_budget / total_water

    # Update all cells simultaneously
    soil = np.clip(
        old_soil + irrigation - ET, 0, 100
    )

# Calculate final fitness
growth = np.mean(
    np.maximum(0, 1 - abs(soil - target) / target)
)

water_used = irrigation.sum()

water_wastage = np.maximum(
    0, soil - target
).sum()

fitness = growth - 0.001 * water_used - 0.001 * water_wastage

# Output
print("Final Soil Moisture:")
print(np.round(soil, 2))

print("\nOptimal Irrigation Quantity:")
print(np.round(irrigation, 2))

print("\nTotal Water Used:", round(water_used, 2))
print("Crop Growth Score:", round(growth, 3))
print("Fitness Value:", round(fitness, 3))


Final Soil Moisture:
[[40. 40. 40.]
 [40. 40. 40.]
 [40. 40. 40.]]

Optimal Irrigation Quantity:
[[2. 2. 2.]
 [2. 2. 2.]
 [2. 2. 2.]]

Total Water Used: 18.0
Crop Growth Score: 1.0
Fitness Value: 0.982
